In [ ]:
# This file compares Variant5 of Algorithm 13 with Variant1 and 2 of Algorithm 13, and with Reviewer t5C9's code.
# The conversion of Reviewer t5C9's code from python to C++, is accomplished by ChatGPT.
# To reproduce the results, please use the kaggle cpu,
# which has 4 CPU cores and 31.35 GB of RAM.


In [ ]:
# List of algorithms (implementations) included in the comparison:
# 1. Reviewer t5C9's code python version
# 2. Reviewer t5C9's code cpp version V1
# 3. Reviewer t5C9's code cpp version V2
# 4. Variant1 of Algorithm 13
# 5. Variant2 of Algorithm 13
# 6. Limited-memory Variant5 of Algorithm 13
# 7. Variant5 of Algorithm 13

In [ ]:
# Difference between Variant5 and Limited-memory Variant5 of Algorithm 13:
# Variant5 of Algorithm 13 fills the MMJ matrix in reconstruction-tree leaf order,
# then permutes it into the original vertex order. Its more sequential writes were faster in our test,
# but the permutation temporarily requires a second full-size matrix.
# Limited-memory Variant5 of Algorithm 13 writes directly to the original vertex positions,
# reducing peak output-matrix memory by roughly half.
# Its scattered writes can be slower, as observed in our test.

In [ ]:
!apt-get update
!apt-get install -y libtbb-dev

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:4 https://cli.github.com/packages stable/main amd64 Packages [359 B]       
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [114 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [11.0 MB]  
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease   
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]    
Get:13 https://r2u.stat.illinois.edu/ubun

In [ ]:
%%writefile Variant1_of_Algorithm_13.cpp

#include <iostream>
#include <vector>
#include <thread>
#include <mutex>
#include <algorithm>
#include <queue>
#include <random>
#include <stack>
#include <chrono>
#include <limits>
#include <tuple>
#include <iomanip>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;

const double INF = numeric_limits<double>::infinity();

// ============================================================
// GRAPH
// ============================================================

struct AdjEdge {
    int to;
    int id;
};

// ============================================================
// PRIM MST
// ============================================================

vector<int> primMST(const Matrix& dist) {

    int n = dist.size();

    vector<double> key(n, INF);
    vector<int> parent(n, -1);
    vector<char> inMST(n, 0);

    priority_queue<
        pair<double,int>,
        vector<pair<double,int>>,
        greater<>
    > pq;

    key[0] = 0.0;

    pq.emplace(0.0, 0);

    while (!pq.empty()) {

        auto [k, u] = pq.top();
        pq.pop();

        if (inMST[u])
            continue;

        inMST[u] = 1;

        const double* row = dist[u].data();

        for (int v = 0; v < n; ++v) {

            double w = row[v];

            if (w && !inMST[v] && w < key[v]) {

                key[v] = w;
                parent[v] = u;

                pq.emplace(w, v);
            }
        }
    }

    return parent;
}

// ============================================================
// BUILD GRAPH
// ============================================================

void buildGraph(
    int n,
    const vector<Edge>& edge_list,
    vector<vector<AdjEdge>>& graph)
{
    graph.assign(n, {});

    for (int i = 0; i < (int)edge_list.size(); ++i) {

        auto [u, v, w] = edge_list[i];

        graph[u].push_back({v, i});
        graph[v].push_back({u, i});
    }
}

// ============================================================
// FAST DFS
// ============================================================

inline void dfs_fast(
    int start,
    const vector<vector<AdjEdge>>& graph,
    const vector<char>& active,
    vector<int>& visited,
    int token,
    vector<int>& nodes)
{
    nodes.clear();

    stack<int> st;

    st.push(start);

    visited[start] = token;

    while (!st.empty()) {

        int u = st.top();
        st.pop();

        nodes.push_back(u);

        for (const auto& e : graph[u]) {

            if (!active[e.id])
                continue;

            int v = e.to;

            if (visited[v] != token) {

                visited[v] = token;

                st.push(v);
            }
        }
    }
}

// ============================================================
// MAIN THREAD
// ============================================================

void main_thread_func(
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int,int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    deque<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    // initially all active
    vector<char> active(num_edges, 1);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;

    while (true) {

        int task;

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.front();
            task_queue.pop_front();
        }

        // remove edge task
        active[task] = 0;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// WORKER
// ============================================================

void worker(
    int tid,
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int,int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    deque<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    vector<char> active(num_edges);

    vector<int> visited(n, -1);

    int token = 0;

    vector<int> tree1, tree2;

    int current_added = num_edges;

    int task;

    fill(active.begin(), active.end(), 0);

    while (true) {

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.back();
            task_queue.pop_back();
        }

        if (task < num_edges - 1){
        for (int i = current_added - 1; i >= task + 1; --i) {
            active[i] = 1;
        }
        }
        current_added = task + 1;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1);

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2);

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// MMJ
// ============================================================

Matrix cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
    const Matrix& distance_matrix,
    int n_jobs)
{


    int n = distance_matrix.size();

    Matrix mmj_matrix(
        n,
        vector<double>(n, 0.0));

    // ========================================================
    // TASK QUEUE
    // ========================================================

    deque<int> task_queue;

    for (int i = 0; i < n - 1; ++i)
        task_queue.push_back(i);

    mutex queue_mutex;

    // ========================================================
    // MST
    // ========================================================

    auto parent = primMST(distance_matrix);

    vector<Edge> edge_list;

    edge_list.reserve(n - 1);

    for (int i = 1; i < n; ++i) {

        edge_list.emplace_back(
            min(i, parent[i]),
            max(i, parent[i]),
            distance_matrix[i][parent[i]]);
    }

    // The distance_matrix can be released after processing it to save memory.

    sort(
        edge_list.begin(),
        edge_list.end(),
        [](const Edge& a, const Edge& b) {

            return get<2>(a) > get<2>(b);
        });

    // ========================================================
    // EDGE ARRAYS
    // ========================================================

    vector<pair<int,int>> edge_nodes;

    vector<double> edge_weights;

    edge_nodes.reserve(n - 1);
    edge_weights.reserve(n - 1);

    for (const auto& [u, v, w] : edge_list) {

        edge_nodes.emplace_back(u, v);

        edge_weights.push_back(w);
    }

    // ========================================================
    // GRAPH
    // ========================================================

    vector<vector<AdjEdge>> graph;

    buildGraph(n, edge_list, graph);

    // ========================================================
    // THREADS
    // ========================================================

    vector<thread> threads;

    for (int t = 0; t < n_jobs - 1; ++t) {

        threads.emplace_back(
            worker,
            t,
            cref(graph),
            cref(edge_nodes),
            cref(edge_weights),
            ref(mmj_matrix),
            ref(task_queue),
            ref(queue_mutex));
    }


    main_thread_func(
        graph,
        edge_nodes,
        edge_weights,
        mmj_matrix,
        task_queue,
        queue_mutex);

    for (auto& th : threads)
        th.join();



    return mmj_matrix;
}

// ============================================================
// DISTANCE MATRIX
// ============================================================

vector<vector<double>> createDistanceMatrix(
    int N,
    int seed)
{
    mt19937 gen(seed);

    uniform_real_distribution<double>
        dist(1.0, 19999.0);

    vector<vector<double>> A(
        N,
        vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {

        for (int j = i + 1; j < N; ++j) {

            double val =
                round(dist(gen) * 100.0) / 100.0;

            A[i][j] = val;
            A[j][i] = val;
        }
    }

    return A;
}


// ============================================================
// MAIN
// ============================================================

int main(int argc, char* argv[]) {

    int N = 1111;
    for (int i = 1; i < argc; ++i) {
        if (std::string(argv[i]) == "--nodes") {
            if (i + 1 >= argc) {
                std::cerr << "--nodes need to be integer\n";
                return 1;
            }
            try {
                N = std::stoi(argv[++i]);
            } catch (const std::exception&) {
                std::cerr << "--nodes must be integer\n";
                return 1;
            }
        }
    }

    int n_jobs =
        thread::hardware_concurrency();

    int random_seed = 2835;

    cout << "Number of nodes: "
         << N << endl;

    cout << "Number of CPU cores: "
         << n_jobs << endl;

    auto distanceMatrix = createDistanceMatrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();

    auto mmjMatrix =
        cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
            distanceMatrix,
            n_jobs);

    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);

    cout << "Time used for MMJ matrix (new implementation of Variant1 of Algorithm 13):"
         << time_used
         << " seconds\n";

    cout << "Print last 30 values of the first row of mmj matrix:\n";

    const auto& row = mmjMatrix[0];

    for (size_t i = row.size() - 30; i < row.size(); ++i)
    {
        cout << fixed
             << setprecision(2)
             << row[i]
             << " ";
    }

    cout << "\n";

    return 0;
}



Writing Variant1_of_Algorithm_13.cpp


In [ ]:
%%writefile Variant2_of_Algorithm_13.cpp

#include <iostream>
#include <vector>
#include <thread>
#include <mutex>
#include <algorithm>
#include <queue>
#include <random>
#include <chrono>
#include <limits>
#include <tuple>
#include <iomanip>
#include <cstdint>

using namespace std;

using Matrix = vector<vector<double>>;
using Edge = tuple<int, int, double>;

const double INF = numeric_limits<double>::infinity();

struct AdjEdge {
    int to;
    int id;
};

// ============================================================
// PRIM MST
// ============================================================

vector<int> primMST(const Matrix& dist) {

    int n = dist.size();

    vector<double> key(n, INF);
    vector<int> parent(n, -1);
    vector<char> inMST(n, 0);

    priority_queue<
        pair<double, int>,
        vector<pair<double, int>>,
        greater<>
    > pq;

    key[0] = 0.0;

    pq.emplace(0.0, 0);

    while (!pq.empty()) {

        auto [k, u] = pq.top();
        pq.pop();

        if (inMST[u])
            continue;

        inMST[u] = 1;

        const double* row = dist[u].data();

        for (int v = 0; v < n; ++v) {

            double w = row[v];

            if (w && !inMST[v] && w < key[v]) {

                key[v] = w;
                parent[v] = u;

                pq.emplace(w, v);
            }
        }
    }

    return parent;
}

// ============================================================
// BUILD GRAPH
// ============================================================

void buildGraph(
    int n,
    const vector<Edge>& edge_list,
    vector<vector<AdjEdge>>& graph)
{
    graph.assign(n, {});

    for (int i = 0; i < (int)edge_list.size(); ++i) {

        auto [u, v, w] = edge_list[i];

        graph[u].push_back({v, i});
        graph[v].push_back({u, i});
    }
}

// ============================================================
// FAST DFS
// ============================================================

inline void dfs_fast(
    int start,
    const vector<vector<AdjEdge>>& graph,
    const vector<char>& active,
    vector<uint32_t>& visited,
    uint32_t token,
    vector<int>& nodes,
    vector<int>& stack_buffer)
{
    nodes.clear();

    stack_buffer.clear();

    stack_buffer.push_back(start);

    visited[start] = token;

    while (!stack_buffer.empty()) {

        int u = stack_buffer.back();
        stack_buffer.pop_back();

        nodes.push_back(u);

        const auto& neighbors = graph[u];

        for (const auto& e : neighbors) {

            if (!active[e.id])
                continue;

            int v = e.to;

            if (visited[v] != token) {

                visited[v] = token;

                stack_buffer.push_back(v);
            }
        }
    }
}

// ============================================================
// WORKER
// ============================================================

void worker(
    int tid,
    const vector<vector<AdjEdge>>& graph,
    const vector<pair<int, int>>& edge_nodes,
    const vector<double>& edge_weights,
    Matrix& mmj_matrix,
    queue<int>& task_queue,
    mutex& queue_mutex)
{
    int n = mmj_matrix.size();

    int num_edges = n - 1;

    // active edges
    vector<char> active(num_edges, 1);

    // IMPORTANT:
    // each worker must progressively remove edges
    int current_removed = -1;

    vector<uint32_t> visited(n, 0);

    uint32_t token = 1;

    vector<int> tree1;
    vector<int> tree2;
    vector<int> stack_buffer;

    tree1.reserve(n);
    tree2.reserve(n);
    stack_buffer.reserve(n);

    while (true) {

        int task;

        {
            lock_guard<mutex> lock(queue_mutex);

            if (task_queue.empty())
                return;

            task = task_queue.front();
            task_queue.pop();
        }

        // ====================================================
        // cumulative edge removals
        // ====================================================

        for (int i = current_removed + 1; i <= task; ++i)
            active[i] = 0;

        current_removed = task;

        auto [u, v] = edge_nodes[task];

        double weight = edge_weights[task];

        // ====================================================
        // DFS 1
        // ====================================================

        ++token;

        dfs_fast(
            u,
            graph,
            active,
            visited,
            token,
            tree1,
            stack_buffer
        );

        // ====================================================
        // DFS 2
        // ====================================================

        ++token;

        dfs_fast(
            v,
            graph,
            active,
            visited,
            token,
            tree2,
            stack_buffer
        );

        // ====================================================
        // fill MMJ matrix
        // ====================================================

        for (int a : tree1) {

            double* rowA = mmj_matrix[a].data();

            for (int b : tree2) {

                rowA[b] = weight;
                mmj_matrix[b][a] = weight;
            }
        }
    }
}

// ============================================================
// MAIN MMJ
// ============================================================

Matrix cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
    const Matrix& distance_matrix,
    int n_jobs)
{
    int n = distance_matrix.size();

    Matrix mmj_matrix(
        n,
        vector<double>(n, 0.0));

    queue<int> task_queue;

    mutex queue_mutex;

    for (int i = 0; i < n - 1; ++i)
        task_queue.push(i);

    // ========================================================
    // MST
    // ========================================================

    auto parent = primMST(distance_matrix);

    // ========================================================
    // EDGE LIST
    // ========================================================

    vector<Edge> edge_list;

    edge_list.reserve(n - 1);

    for (int i = 1; i < n; ++i) {

        int p = parent[i];

        edge_list.emplace_back(
            min(i, p),
            max(i, p),
            distance_matrix[i][p]
        );
    }

    // The distance_matrix can be released after processing it to save memory.


    // descending order
    sort(edge_list.begin(),
         edge_list.end(),
         [](const Edge& a, const Edge& b) {
             return get<2>(a) > get<2>(b);
         });

    // ========================================================
    // EDGE ARRAYS
    // ========================================================

    vector<pair<int, int>> edge_nodes;
    vector<double> edge_weights;

    edge_nodes.reserve(n - 1);
    edge_weights.reserve(n - 1);

    for (const auto& [u, v, w] : edge_list) {

        edge_nodes.emplace_back(u, v);
        edge_weights.push_back(w);
    }

    // ========================================================
    // GRAPH
    // IMPORTANT:
    // must build AFTER sorting edge_list
    // so edge IDs match task IDs
    // ========================================================

    vector<vector<AdjEdge>> graph;

    buildGraph(n, edge_list, graph);

    // ========================================================
    // THREADS
    // ========================================================

    vector<thread> threads;

    for (int t = 0; t < n_jobs; ++t) {

        threads.emplace_back(
            worker,
            t,
            cref(graph),
            cref(edge_nodes),
            cref(edge_weights),
            ref(mmj_matrix),
            ref(task_queue),
            ref(queue_mutex)
        );
    }

    for (auto& th : threads)
        th.join();

    return mmj_matrix;
}

// ============================================================
// DISTANCE MATRIX
// ============================================================

Matrix createDistanceMatrix(
    int N,
    int seed)
{
    mt19937 gen(seed);

    uniform_real_distribution<double>
        dist(1.0, 19999.0);

    Matrix A(
        N,
        vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {

        for (int j = i + 1; j < N; ++j) {

            double val =
                round(dist(gen) * 100.0) / 100.0;

            A[i][j] = val;
            A[j][i] = val;
        }
    }

    return A;
}

// ============================================================
// MAIN
// ============================================================
int main(int argc, char* argv[]) {

    // WARNING:
    // Dense matrices explode in memory quickly
    int N = 1111;
    for (int i = 1; i < argc; ++i) {
        if (std::string(argv[i]) == "--nodes") {
            if (i + 1 >= argc) {
                std::cerr << "--nodes need to be integer\n";
                return 1;
            }
            try {
                N = std::stoi(argv[++i]);
            } catch (const std::exception&) {
                std::cerr << "--nodes must be integer\n";
                return 1;
            }
        }
    }


    int n_jobs =
        thread::hardware_concurrency();

    int random_seed = 2835;

    cout << "Number of nodes: "
         << N << endl;

    cout << "Number of CPU cores: "
         << n_jobs << endl;

    auto distanceMatrix =
        createDistanceMatrix(
            N,
            random_seed
        );

    auto start = chrono::high_resolution_clock::now();

    auto mmjMatrix =
        cal_mmj_matrix_by_algo_4_Calculation_and_Copy_parallel_compu(
            distanceMatrix,
            n_jobs
        );

    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed
         << setprecision(3);

    cout << "Time used for MMJ matrix (new implementation of Variant2 of Algorithm 13):"
         << time_used
         << " seconds\n";

    cout << "Print last 30 values of first row of mmj matrix:\n";

    const auto& row = mmjMatrix[0];

    for (size_t i = row.size() - 30;
         i < row.size();
         ++i)
    {
        cout << fixed
             << setprecision(2)
             << row[i]
             << " ";
    }

    cout << "\n";

    return 0;
}


Writing Variant2_of_Algorithm_13.cpp


In [ ]:
%%writefile Reviewer_t5C9_code_cpp_version_V1.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <limits>
#include <random>
#include <chrono>
#include <algorithm>
#include <list>
#include <iomanip>



using namespace std;

struct Edge {
    int u, v;
    double weight;
};

// Prim's MST (dense graph version)
vector<Edge> prim_mst(const vector<vector<double>>& D) {
    int n = D.size();
    vector<bool> in_mst(n, false);
    vector<double> key(n, numeric_limits<double>::infinity());
    vector<int> parent(n, -1);
    key[0] = 0.0;

    for (int count = 0; count < n; ++count) {
        double min_val = numeric_limits<double>::infinity();
        int u = -1;
        for (int i = 0; i < n; ++i)
            if (!in_mst[i] && key[i] < min_val)
                min_val = key[i], u = i;

        if (u == -1) break;

        in_mst[u] = true;
        for (int v = 0; v < n; ++v)
            if (!in_mst[v] && D[u][v] < key[v])
                key[v] = D[u][v], parent[v] = u;
    }

    vector<Edge> edges;
    for (int v = 1; v < n; ++v)
        edges.push_back({parent[v], v, D[parent[v]][v]});
    return edges;
}

// Build CSR-like structure
void build_csr(int n, const vector<Edge>& mst_edges,
               vector<int>& ptr, vector<int>& adj_edges, vector<double>& adj_weights) {
    vector<int> edge_counts(n, 0);
    for (const auto& e : mst_edges) {
        edge_counts[e.u]++;
        edge_counts[e.v]++;
    }

    ptr.resize(n + 1);
    for (int i = 1; i <= n; ++i)
        ptr[i] = ptr[i - 1] + edge_counts[i - 1];

    adj_edges.resize(ptr[n]);
    adj_weights.resize(ptr[n]);
    vector<int> positions(n, 0);

    for (const auto& e : mst_edges) {
        for (int i = 0; i < 2; ++i) {
            int u = (i == 0) ? e.u : e.v;
            int v = (i == 0) ? e.v : e.u;
            int idx = ptr[u] + positions[u]++;
            adj_edges[idx] = v;
            adj_weights[idx] = e.weight;
        }
    }
}


#include <tbb/parallel_for.h>
#include <tbb/blocked_range.h>
#include <tbb/parallel_for_each.h>

vector<vector<double>> compute_bottleneck_matrix(int n,
    const vector<int>& ptr, const vector<int>& adj_edges, const vector<double>& adj_weights) {

    vector<vector<double>> bottleneck(n, vector<double>(n, 0.0));

    tbb::parallel_for(tbb::blocked_range<int>(0, n),
        [&](const tbb::blocked_range<int>& r) {
            for (int src = r.begin(); src < r.end(); ++src) {
                vector<bool> visited(n, false);
                vector<double> max_edges(n, 0.0);
                queue<pair<int, double>> q;

                visited[src] = true;
                q.push({src, 0.0});

                while (!q.empty()) {
                    auto [u, curr_max] = q.front(); q.pop();

                    for (int i = ptr[u]; i < ptr[u + 1]; ++i) {
                        int v = adj_edges[i];
                        double weight = adj_weights[i];

                        if (!visited[v]) {
                            double new_max = max(curr_max, weight);
                            visited[v] = true;
                            max_edges[v] = new_max;
                            q.push({v, new_max});
                        }
                    }
                }

                bottleneck[src] = move(max_edges); // safe: each src owns its row
            }
        }
    );

    return bottleneck;
}


vector<vector<double>> ultra_fast_wide(const vector<vector<double>>& D) {
    int n = D.size();


    vector<Edge> mst = prim_mst(D);

    vector<int> ptr, adj_edges;
    vector<double> adj_weights;
    build_csr(n, mst, ptr, adj_edges, adj_weights);

    return compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights);
}

vector<vector<double>> create_symmetric_distance_matrix(int N, int seed) {
    mt19937 gen(seed);

    // Generate doubles between 1.00 and 19999.00
    uniform_real_distribution<double> dist(1.0, 19999.0);

    vector<vector<double>> A(N, vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {
        for (int j = i + 1; j < N; ++j) {

            // Round to 2 decimal places
            double val = round(dist(gen) * 100.0) / 100.0;

            A[i][j] = A[j][i] = val;
        }
    }

    return A;
}



int main(int argc, char* argv[]) {

    int N = 1111;
    for (int i = 1; i < argc; ++i) {
        if (std::string(argv[i]) == "--nodes") {
            if (i + 1 >= argc) {
                std::cerr << "--nodes need to be integer\n";
                return 1;
            }
            try {
                N = std::stoi(argv[++i]);
            } catch (const std::exception&) {
                std::cerr << "--nodes must be integer\n";
                return 1;
            }
        }
    }

    int n_jobs = thread::hardware_concurrency();
    int random_seed = 2835;


    cout << "Number of nodes: "<< N << endl;
    cout << "Number of CPU cores: " << n_jobs << endl;



    auto distanceMatrix = create_symmetric_distance_matrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();
    auto mmjMatrix = ultra_fast_wide(distanceMatrix);
    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);
    cout << "Time used for MMJ matrix (Reviewer t5C9's code cpp version - V1): " << time_used << " seconds\n";
    cout << "Print last 30 values of the first row of mmj matrix: " << endl;
    const auto& row = mmjMatrix[0];
    for (size_t i = row.size() - 30; i < row.size(); ++i)
        cout << fixed << setprecision(2) << row[i] << " ";
    cout << "\n";

    return 0;

}



Writing Reviewer_t5C9_code_cpp_version_V1.cpp


In [ ]:
%%writefile Variant5_of_Algorithm_13.cpp
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstddef>
#include <cstdint>
#include <cstdlib>
#include <functional>
#include <iomanip>
#include <iostream>
#include <limits>
#include <memory>
#include <numeric>
#include <queue>
#include <random>
#include <stdexcept>
#include <string>
#include <thread>
#include <tuple>
#include <utility>
#include <vector>



// Pipeline:
//   1. Compute an MST of the dense input graph.
//   2. Build a Kruskal reconstruction tree from the MST.
//   3. Put the original vertices in reconstruction-tree leaf order.  Every
//      reconstruction-tree subtree is then a contiguous leaf interval.
//   4. An internal reconstruction-tree node with children A and B says that
//      every pair in A x B has minimax distance equal to that node's weight.
//      Fill those two rectangular matrix regions.
//   5. Threads own disjoint output-row ranges.  Consequently no output lock,
//      atomic operation, or concurrent write to the same cache line is needed
//      in the main filling phase.
//   6. Optionally restore the matrix to the original vertex numbering.

namespace mmj {

using Clock = std::chrono::steady_clock;

double seconds_between(Clock::time_point a, Clock::time_point b) {
    return std::chrono::duration<double>(b - a).count();
}

class DenseMatrix {
public:
    DenseMatrix() = default;

    // Memory is intentionally uninitialized unless zero_initialize is true.
    // The MMJ filling algorithm writes every off-diagonal entry exactly once
    // and explicitly writes every diagonal entry.
    explicit DenseMatrix(std::size_t n, bool zero_initialize = false)
        : n_(n) {
        if (n_ != 0 && n_ > std::numeric_limits<std::size_t>::max() / n_) {
            throw std::overflow_error("matrix element count overflow");
        }
        const std::size_t count = n_ * n_;
        if (count != 0) {
            data_ = std::unique_ptr<double[]>(new double[count]);
            if (zero_initialize) {
                std::fill_n(data_.get(), count, 0.0);
            }
        }
    }

    DenseMatrix(DenseMatrix&&) noexcept = default;
    DenseMatrix& operator=(DenseMatrix&&) noexcept = default;
    DenseMatrix(const DenseMatrix&) = delete;
    DenseMatrix& operator=(const DenseMatrix&) = delete;

    std::size_t n() const noexcept { return n_; }
    std::size_t elements() const noexcept { return n_ * n_; }

    double* row(std::size_t i) noexcept { return data_.get() + i * n_; }
    const double* row(std::size_t i) const noexcept {
        return data_.get() + i * n_;
    }

    double& operator()(std::size_t i, std::size_t j) noexcept {
        return data_[i * n_ + j];
    }
    const double& operator()(std::size_t i, std::size_t j) const noexcept {
        return data_[i * n_ + j];
    }

    void release() noexcept {
        data_.reset();
        n_ = 0;
    }

private:
    std::size_t n_ = 0;
    std::unique_ptr<double[]> data_;
};

struct Edge {
    int u;
    int v;
    double weight;
};

// Heap-based Prim for a dense matrix.  It scans one dense row for each chosen
// vertex, while the heap avoids a second O(n^2) scan for the next vertex.
std::vector<Edge> prim_mst_dense(const DenseMatrix& distance) {
    const std::size_t n_size = distance.n();
    if (n_size == 0) {
        return {};
    }
    if (n_size > static_cast<std::size_t>(std::numeric_limits<int>::max())) {
        throw std::invalid_argument("too many vertices for 32-bit indices");
    }
    const int n = static_cast<int>(n_size);
    const double inf = std::numeric_limits<double>::infinity();

    std::vector<double> key(n, inf);
    std::vector<int> parent(n, -1);
    std::vector<std::uint8_t> in_mst(n, 0);

    using HeapItem = std::pair<double, int>;
    std::priority_queue<HeapItem, std::vector<HeapItem>,
                        std::greater<HeapItem>> heap;
    key[0] = 0.0;
    heap.emplace(0.0, 0);

    int chosen = 0;
    while (!heap.empty()) {
        const auto [ignored_key, u] = heap.top();
        (void)ignored_key;
        heap.pop();
        if (in_mst[u]) {
            continue;
        }
        in_mst[u] = 1;
        ++chosen;

        const double* drow = distance.row(static_cast<std::size_t>(u));
        for (int v = 0; v < n; ++v) {
            const double w = drow[v];
            if (!in_mst[v] && w < key[v]) {
                key[v] = w;
                parent[v] = u;
                heap.emplace(w, v);
            }
        }
    }

    if (chosen != n) {
        throw std::runtime_error("input graph is disconnected");
    }

    std::vector<Edge> mst;
    mst.reserve(n > 0 ? static_cast<std::size_t>(n - 1) : 0);
    for (int v = 1; v < n; ++v) {
        if (parent[v] < 0) {
            throw std::runtime_error("Prim failed to assign a parent");
        }
        mst.push_back({parent[v], v, distance(v, parent[v])});
    }
    return mst;
}

class DisjointSet {
public:
    explicit DisjointSet(int n)
        : parent_(n), size_(n, 1), tree_root_(n) {
        std::iota(parent_.begin(), parent_.end(), 0);
        std::iota(tree_root_.begin(), tree_root_.end(), 0);
    }

    int find(int x) {
        int root = x;
        while (parent_[root] != root) {
            root = parent_[root];
        }
        while (parent_[x] != x) {
            const int next = parent_[x];
            parent_[x] = root;
            x = next;
        }
        return root;
    }

    int tree_root_of_set(int set_root) const { return tree_root_[set_root]; }

    int unite_roots(int a, int b, int new_tree_root) {
        if (size_[a] < size_[b]) {
            std::swap(a, b);
        }
        parent_[b] = a;
        size_[a] += size_[b];
        tree_root_[a] = new_tree_root;
        return a;
    }

private:
    std::vector<int> parent_;
    std::vector<int> size_;
    std::vector<int> tree_root_;
};

struct ReconstructionNode {
    int left = -1;
    int right = -1;
    double weight = 0.0;
    int begin = -1;  // inclusive leaf-order position
    int end = -1;    // exclusive leaf-order position
};

struct ReconstructionTree {
    int original_vertex_count = 0;
    int root = -1;
    std::vector<ReconstructionNode> nodes;
    // leaf_order[position] = original vertex id
    std::vector<int> leaf_order;
    // position[original vertex id] = position in leaf_order
    std::vector<int> position;
};

ReconstructionTree build_reconstruction_tree(
    int n, std::vector<Edge> mst_edges) {
    if (n <= 0) {
        return {};
    }
    if (static_cast<int>(mst_edges.size()) != n - 1) {
        throw std::invalid_argument("an n-vertex MST must contain n-1 edges");
    }

    std::sort(mst_edges.begin(), mst_edges.end(),
              [](const Edge& a, const Edge& b) {
                  if (a.weight != b.weight) return a.weight < b.weight;
                  if (a.u != b.u) return a.u < b.u;
                  return a.v < b.v;
              });

    ReconstructionTree tree;
    tree.original_vertex_count = n;
    tree.nodes.resize(static_cast<std::size_t>(2 * n - 1));
    tree.leaf_order.resize(n);
    tree.position.resize(n);

    DisjointSet dsu(n);
    int next_node = n;

    for (const Edge& edge : mst_edges) {
        int a = dsu.find(edge.u);
        int b = dsu.find(edge.v);
        if (a == b) {
            throw std::runtime_error("input edge set is not a tree");
        }

        const int left_root = dsu.tree_root_of_set(a);
        const int right_root = dsu.tree_root_of_set(b);
        ReconstructionNode& node = tree.nodes[next_node];
        node.left = left_root;
        node.right = right_root;
        node.weight = edge.weight;

        dsu.unite_roots(a, b, next_node);
        ++next_node;
    }

    tree.root = dsu.tree_root_of_set(dsu.find(0));

    // Iterative postorder traversal: safe even for a completely skewed tree.
    std::vector<std::pair<int, bool>> stack;
    stack.reserve(static_cast<std::size_t>(4 * n));
    stack.emplace_back(tree.root, false);
    int cursor = 0;

    while (!stack.empty()) {
        const auto [node_id, expanded] = stack.back();
        stack.pop_back();
        ReconstructionNode& node = tree.nodes[node_id];

        if (node_id < n) {
            node.begin = cursor;
            node.end = cursor + 1;
            tree.leaf_order[cursor] = node_id;
            tree.position[node_id] = cursor;
            ++cursor;
            continue;
        }

        if (!expanded) {
            stack.emplace_back(node_id, true);
            // Push right first so that left is processed first.
            stack.emplace_back(node.right, false);
            stack.emplace_back(node.left, false);
        } else {
            const ReconstructionNode& left = tree.nodes[node.left];
            const ReconstructionNode& right = tree.nodes[node.right];
            node.begin = left.begin;
            node.end = right.end;
            if (left.end != right.begin) {
                throw std::runtime_error("non-contiguous reconstruction subtree");
            }
        }
    }

    if (cursor != n) {
        throw std::runtime_error("reconstruction tree does not contain all leaves");
    }
    return tree;
}

unsigned normalized_thread_count(unsigned requested, std::size_t n) {
    unsigned threads = requested;
    if (threads == 0) {
        threads = std::thread::hardware_concurrency();
    }
    if (threads == 0) {
        threads = 1;
    }
    if (n != 0) {
        threads = std::min<unsigned>(threads, static_cast<unsigned>(n));
    }
    return std::max(1u, threads);
}

template <class Function>
void parallel_rows(std::size_t n, unsigned requested_threads, Function fn) {
    const unsigned threads = normalized_thread_count(requested_threads, n);
    std::vector<std::thread> workers;
    workers.reserve(threads > 0 ? threads - 1 : 0);

    auto run = [&](unsigned tid) {
        const std::size_t begin = n * tid / threads;
        const std::size_t end = n * (tid + 1) / threads;
        fn(begin, end);
    };

    // The caller thread owns the last partition.  Total active computation
    // threads therefore equals 'threads', not threads+1.
    for (unsigned tid = 0; tid + 1 < threads; ++tid) {
        workers.emplace_back(run, tid);
    }
    run(threads - 1);
    for (std::thread& worker : workers) {
        worker.join();
    }
}

DenseMatrix fill_in_leaf_order(const ReconstructionTree& tree,
                               unsigned requested_threads) {
    const int n = tree.original_vertex_count;
    DenseMatrix output(static_cast<std::size_t>(n), false);

    parallel_rows(static_cast<std::size_t>(n), requested_threads,
                  [&](std::size_t owned_begin, std::size_t owned_end) {
        // Every thread owns complete rows.  It scans the small O(n) list of
        // internal nodes and writes only intersections with its row range.
        for (int node_id = n; node_id < 2 * n - 1; ++node_id) {
            const ReconstructionNode& node = tree.nodes[node_id];
            const ReconstructionNode& left = tree.nodes[node.left];
            const ReconstructionNode& right = tree.nodes[node.right];

            const std::size_t left_begin = static_cast<std::size_t>(left.begin);
            const std::size_t left_end = static_cast<std::size_t>(left.end);
            const std::size_t right_begin = static_cast<std::size_t>(right.begin);
            const std::size_t right_end = static_cast<std::size_t>(right.end);

            // Owned rows in the left child: fill their right-child interval.
            const std::size_t lr_begin = std::max(owned_begin, left_begin);
            const std::size_t lr_end = std::min(owned_end, left_end);
            for (std::size_t row = lr_begin; row < lr_end; ++row) {
                std::fill(output.row(row) + right_begin,
                          output.row(row) + right_end, node.weight);
            }

            // Owned rows in the right child: fill their left-child interval.
            const std::size_t rr_begin = std::max(owned_begin, right_begin);
            const std::size_t rr_end = std::min(owned_end, right_end);
            for (std::size_t row = rr_begin; row < rr_end; ++row) {
                std::fill(output.row(row) + left_begin,
                          output.row(row) + left_end, node.weight);
            }
        }

        for (std::size_t row = owned_begin; row < owned_end; ++row) {
            output(row, row) = 0.0;
        }
    });
    return output;
}

// Convert P[leaf_position(i), leaf_position(j)] into M[i, j].
// Each source and destination row stays private to one worker.  Within a row,
// the source is read sequentially; the destination permutation usually fits
// in the core's private cache for the graph sizes targeted here.
DenseMatrix restore_original_order(const DenseMatrix& permuted,
                                   const ReconstructionTree& tree,
                                   unsigned requested_threads) {
    const std::size_t n = permuted.n();
    DenseMatrix restored(n, false);

    parallel_rows(n, requested_threads,
                  [&](std::size_t original_begin, std::size_t original_end) {
        for (std::size_t original_row = original_begin;
             original_row < original_end; ++original_row) {
            const std::size_t source_position = static_cast<std::size_t>(
                tree.position[original_row]);
            const double* source = permuted.row(source_position);
            double* destination = restored.row(original_row);

            for (std::size_t leaf_position = 0; leaf_position < n;
                 ++leaf_position) {
                const std::size_t original_column = static_cast<std::size_t>(
                    tree.leaf_order[leaf_position]);
                destination[original_column] = source[leaf_position];
            }
        }
    });
    return restored;
}

struct Result {
    DenseMatrix matrix;
    ReconstructionTree reconstruction_tree;
    bool in_original_order = false;
};

Result compute_from_mst(int n, const std::vector<Edge>& mst,
                        unsigned requested_threads,
                        bool restore_vertex_order) {
    ReconstructionTree tree = build_reconstruction_tree(n, mst);
    DenseMatrix permuted = fill_in_leaf_order(tree, requested_threads);

    DenseMatrix final_matrix;
    if (restore_vertex_order) {
        final_matrix = restore_original_order(permuted, tree, requested_threads);
        permuted.release();
    } else {
        final_matrix = std::move(permuted);
    }

    return {std::move(final_matrix), std::move(tree), restore_vertex_order};
}

DenseMatrix create_symmetric_distance_matrix(int n, std::uint32_t seed) {
    if (n < 0) {
        throw std::invalid_argument("n must be nonnegative");
    }
    DenseMatrix matrix(static_cast<std::size_t>(n), false);
    std::mt19937 generator(seed);
    std::uniform_real_distribution<double> distribution(1.0, 19999.0);

    for (int i = 0; i < n; ++i) {
        matrix(i, i) = 0.0;
        for (int j = i + 1; j < n; ++j) {
            const double value = std::round(distribution(generator) * 100.0) /
                                 100.0;
            matrix(i, j) = value;
            matrix(j, i) = value;
        }
    }
    return matrix;
}

double value_in_original_numbering(const Result& result, int i, int j) {
    if (result.in_original_order) {
        return result.matrix(i, j);
    }
    const int pi = result.reconstruction_tree.position[i];
    const int pj = result.reconstruction_tree.position[j];
    return result.matrix(pi, pj);
}

}  // namespace mmj

int main(int argc, char* argv[]) {

    int N = 1111;
    for (int i = 1; i < argc; ++i) {
        if (std::string(argv[i]) == "--nodes") {
            if (i + 1 >= argc) {
                std::cerr << "--nodes need to be integer\n";
                return 1;
            }
            try {
                N = std::stoi(argv[++i]);
            } catch (const std::exception&) {
                std::cerr << "--nodes must be integer\n";
                return 1;
            }
        }
    }

    unsigned n_jobs = std::thread::hardware_concurrency();
    if (n_jobs == 0) {
        n_jobs = 1;
    }

    const std::uint32_t random_seed = 2835;
    const bool restore_original_order = true;

    std::cout << "Number of nodes: "
              << N << '\n';

    std::cout << "Number of CPU cores: "
              << mmj::normalized_thread_count(n_jobs, N) << '\n';

    auto distance_matrix =
        mmj::create_symmetric_distance_matrix(
            N,
            random_seed
        );

    const auto start = mmj::Clock::now();

    auto mst =
        mmj::prim_mst_dense(
            distance_matrix
        );

    distance_matrix.release();

    auto result =
        mmj::compute_from_mst(
            N,
            mst,
            n_jobs,
            restore_original_order
        );

    const auto end = mmj::Clock::now();

    const double time_used =
        mmj::seconds_between(
            start,
            end
        );

    std::cout << std::fixed
              << std::setprecision(3);

    std::cout << "Time used for MMJ matrix "
                 "(Variant5 of Algorithm 13): "
              << time_used
              << " seconds\n";

    std::cout << "Print last 30 values of first row "
                 "of MMJ matrix:\n";

    const int first = std::max(0, N - 30);

    for (int j = first; j < N; ++j) {
        std::cout << std::fixed
                  << std::setprecision(2)
                  << mmj::value_in_original_numbering(
                         result,
                         0,
                         j
                     )
                  << " ";
    }

    std::cout << "\n";

    return 0;
}




Writing Variant5_of_Algorithm_13.cpp


In [ ]:
%%writefile Reviewer_t5C9_code_cpp_version_V1.cpp

#include <iostream>
#include <vector>
#include <queue>
#include <limits>
#include <random>
#include <chrono>
#include <algorithm>
#include <list>
#include <iomanip>



using namespace std;

struct Edge {
    int u, v;
    double weight;
};

// Prim's MST (dense graph version)
vector<Edge> prim_mst(const vector<vector<double>>& D) {
    int n = D.size();
    vector<bool> in_mst(n, false);
    vector<double> key(n, numeric_limits<double>::infinity());
    vector<int> parent(n, -1);
    key[0] = 0.0;

    for (int count = 0; count < n; ++count) {
        double min_val = numeric_limits<double>::infinity();
        int u = -1;
        for (int i = 0; i < n; ++i)
            if (!in_mst[i] && key[i] < min_val)
                min_val = key[i], u = i;

        if (u == -1) break;

        in_mst[u] = true;
        for (int v = 0; v < n; ++v)
            if (!in_mst[v] && D[u][v] < key[v])
                key[v] = D[u][v], parent[v] = u;
    }

    vector<Edge> edges;
    for (int v = 1; v < n; ++v)
        edges.push_back({parent[v], v, D[parent[v]][v]});
    return edges;
}

// Build CSR-like structure
void build_csr(int n, const vector<Edge>& mst_edges,
               vector<int>& ptr, vector<int>& adj_edges, vector<double>& adj_weights) {
    vector<int> edge_counts(n, 0);
    for (const auto& e : mst_edges) {
        edge_counts[e.u]++;
        edge_counts[e.v]++;
    }

    ptr.resize(n + 1);
    for (int i = 1; i <= n; ++i)
        ptr[i] = ptr[i - 1] + edge_counts[i - 1];

    adj_edges.resize(ptr[n]);
    adj_weights.resize(ptr[n]);
    vector<int> positions(n, 0);

    for (const auto& e : mst_edges) {
        for (int i = 0; i < 2; ++i) {
            int u = (i == 0) ? e.u : e.v;
            int v = (i == 0) ? e.v : e.u;
            int idx = ptr[u] + positions[u]++;
            adj_edges[idx] = v;
            adj_weights[idx] = e.weight;
        }
    }
}


#include <tbb/parallel_for.h>
#include <tbb/blocked_range.h>
#include <tbb/parallel_for_each.h>

vector<vector<double>> compute_bottleneck_matrix(int n,
    const vector<int>& ptr, const vector<int>& adj_edges, const vector<double>& adj_weights) {

    vector<vector<double>> bottleneck(n, vector<double>(n, 0.0));

    tbb::parallel_for(tbb::blocked_range<int>(0, n),
        [&](const tbb::blocked_range<int>& r) {
            for (int src = r.begin(); src < r.end(); ++src) {
                vector<bool> visited(n, false);
                vector<double> max_edges(n, 0.0);
                queue<pair<int, double>> q;

                visited[src] = true;
                q.push({src, 0.0});

                while (!q.empty()) {
                    auto [u, curr_max] = q.front(); q.pop();

                    for (int i = ptr[u]; i < ptr[u + 1]; ++i) {
                        int v = adj_edges[i];
                        double weight = adj_weights[i];

                        if (!visited[v]) {
                            double new_max = max(curr_max, weight);
                            visited[v] = true;
                            max_edges[v] = new_max;
                            q.push({v, new_max});
                        }
                    }
                }

                bottleneck[src] = move(max_edges); // safe: each src owns its row
            }
        }
    );

    return bottleneck;
}


vector<vector<double>> ultra_fast_wide(const vector<vector<double>>& D) {
    int n = D.size();


    vector<Edge> mst = prim_mst(D);

    vector<int> ptr, adj_edges;
    vector<double> adj_weights;
    build_csr(n, mst, ptr, adj_edges, adj_weights);

    return compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights);
}

vector<vector<double>> create_symmetric_distance_matrix(int N, int seed) {
    mt19937 gen(seed);

    // Generate doubles between 1.00 and 19999.00
    uniform_real_distribution<double> dist(1.0, 19999.0);

    vector<vector<double>> A(N, vector<double>(N, 0.0));

    for (int i = 0; i < N; ++i) {
        for (int j = i + 1; j < N; ++j) {

            // Round to 2 decimal places
            double val = round(dist(gen) * 100.0) / 100.0;

            A[i][j] = A[j][i] = val;
        }
    }

    return A;
}



int main(int argc, char* argv[]) {

    int N = 1111;
    for (int i = 1; i < argc; ++i) {
        if (std::string(argv[i]) == "--nodes") {
            if (i + 1 >= argc) {
                std::cerr << "--nodes need to be integer\n";
                return 1;
            }
            try {
                N = std::stoi(argv[++i]);
            } catch (const std::exception&) {
                std::cerr << "--nodes must be integer\n";
                return 1;
            }
        }
    }

    int n_jobs = thread::hardware_concurrency();
    int random_seed = 2835;


    cout << "Number of nodes: "<< N << endl;
    cout << "Number of CPU cores: " << n_jobs << endl;



    auto distanceMatrix = create_symmetric_distance_matrix(N, random_seed);

    auto start = chrono::high_resolution_clock::now();
    auto mmjMatrix = ultra_fast_wide(distanceMatrix);
    auto end = chrono::high_resolution_clock::now();

    double time_used = chrono::duration<double>(end - start).count();

    cout << fixed << setprecision(3);
    cout << "Time used for MMJ matrix (Reviewer t5C9's code cpp version - V1): " << time_used << " seconds\n";
    cout << "Print last 30 values of the first row of mmj matrix: " << endl;
    const auto& row = mmjMatrix[0];
    for (size_t i = row.size() - 30; i < row.size(); ++i)
        cout << fixed << setprecision(2) << row[i] << " ";
    cout << "\n";

    return 0;

}

Overwriting Reviewer_t5C9_code_cpp_version_V1.cpp


In [ ]:
%%writefile Reviewer_t5C9_code_cpp_version_V2.cpp


#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstddef>
#include <cstdint>
#include <cstdlib>
#include <functional>
#include <iomanip>
#include <iostream>
#include <limits>
#include <memory>
#include <numeric>
#include <queue>
#include <random>
#include <stdexcept>
#include <string>
#include <thread>
#include <utility>
#include <vector>

// Optimized implementation of Reviewer t5C9's algorithm.
//
// The algorithm itself is unchanged:
//   1. Compute an MST of the dense distance matrix.
//   2. For every source vertex, traverse the entire MST.
//   3. Propagate the maximum edge weight on the source-to-vertex path.
//
// Implementation improvements over the original C++ version:
//   * one flat contiguous output allocation instead of vector<vector<double>>;
//   * compact CSR representation of the MST;
//   * no vector<bool> visited array (an MST is a tree, so parent is sufficient);
//   * no max_edges temporary row and no subsequent row move/copy;
//   * no std::queue<pair<int,double>> and no per-source heap allocations;
//   * one reusable vector<StackItem> per worker;
//   * static row ownership: every worker writes complete, disjoint rows;
//   * exactly the requested number of computation threads, including main.

namespace reviewer_t5c9 {

using Clock = std::chrono::steady_clock;

double seconds_between(Clock::time_point begin, Clock::time_point end) {
    return std::chrono::duration<double>(end - begin).count();
}

class DenseMatrix {
public:
    DenseMatrix() = default;

    // Memory is uninitialized unless zero_initialize is true.  The calculation
    // writes every output element, so zeroing a multi-gigabyte matrix first
    // would be pure overhead.
    explicit DenseMatrix(std::size_t n, bool zero_initialize = false)
        : n_(n) {
        if (n_ != 0 && n_ > std::numeric_limits<std::size_t>::max() / n_) {
            throw std::overflow_error("matrix element count overflow");
        }
        const std::size_t count = n_ * n_;
        if (count != 0) {
            data_ = std::unique_ptr<double[]>(new double[count]);
            if (zero_initialize) {
                std::fill_n(data_.get(), count, 0.0);
            }
        }
    }

    DenseMatrix(DenseMatrix&&) noexcept = default;
    DenseMatrix& operator=(DenseMatrix&&) noexcept = default;
    DenseMatrix(const DenseMatrix&) = delete;
    DenseMatrix& operator=(const DenseMatrix&) = delete;

    std::size_t n() const noexcept { return n_; }

    double* row(std::size_t i) noexcept { return data_.get() + i * n_; }
    const double* row(std::size_t i) const noexcept {
        return data_.get() + i * n_;
    }

    double& operator()(std::size_t i, std::size_t j) noexcept {
        return data_[i * n_ + j];
    }
    const double& operator()(std::size_t i, std::size_t j) const noexcept {
        return data_[i * n_ + j];
    }

    void release() noexcept {
        data_.reset();
        n_ = 0;
    }

private:
    std::size_t n_ = 0;
    std::unique_ptr<double[]> data_;
};

struct Edge {
    int u;
    int v;
    double weight;
};

// Heap-based Prim suited to the notebook's dense complete graph.  It scans
// exactly one dense matrix row for each selected vertex and avoids the extra
// O(n^2) linear search for the next minimum-key vertex.
std::vector<Edge> prim_mst_dense(const DenseMatrix& distance) {
    const std::size_t n_size = distance.n();
    if (n_size == 0) return {};
    if (n_size > static_cast<std::size_t>(std::numeric_limits<int>::max())) {
        throw std::invalid_argument("too many vertices for 32-bit indices");
    }

    const int n = static_cast<int>(n_size);
    const double infinity = std::numeric_limits<double>::infinity();
    std::vector<double> key(n, infinity);
    std::vector<int> parent(n, -1);
    std::vector<std::uint8_t> in_mst(n, 0);

    using HeapItem = std::pair<double, int>;
    std::priority_queue<HeapItem, std::vector<HeapItem>,
                        std::greater<HeapItem>> heap;
    key[0] = 0.0;
    heap.emplace(0.0, 0);

    int selected = 0;
    while (!heap.empty()) {
        const auto [ignored_key, u] = heap.top();
        (void)ignored_key;
        heap.pop();
        if (in_mst[u]) continue;

        in_mst[u] = 1;
        ++selected;
        const double* input_row = distance.row(static_cast<std::size_t>(u));

        for (int v = 0; v < n; ++v) {
            const double weight = input_row[v];
            if (!in_mst[v] && weight < key[v]) {
                key[v] = weight;
                parent[v] = u;
                heap.emplace(weight, v);
            }
        }
    }

    if (selected != n) {
        throw std::runtime_error("input graph is disconnected");
    }

    std::vector<Edge> mst;
    mst.reserve(static_cast<std::size_t>(n - 1));
    for (int v = 1; v < n; ++v) {
        if (parent[v] < 0) {
            throw std::runtime_error("Prim failed to assign a parent");
        }
        mst.push_back({parent[v], v, distance(v, parent[v])});
    }
    return mst;
}

struct CsrTree {
    std::vector<int> offset;
    std::vector<int> neighbor;
    std::vector<double> weight;
};

CsrTree build_csr_tree(int n, const std::vector<Edge>& mst) {
    if (n <= 0 || static_cast<int>(mst.size()) != n - 1) {
        throw std::invalid_argument("invalid MST size");
    }

    CsrTree tree;
    tree.offset.assign(static_cast<std::size_t>(n + 1), 0);

    for (const Edge& edge : mst) {
        ++tree.offset[static_cast<std::size_t>(edge.u + 1)];
        ++tree.offset[static_cast<std::size_t>(edge.v + 1)];
    }
    for (int i = 1; i <= n; ++i) {
        tree.offset[i] += tree.offset[i - 1];
    }

    tree.neighbor.resize(static_cast<std::size_t>(2 * (n - 1)));
    tree.weight.resize(static_cast<std::size_t>(2 * (n - 1)));
    std::vector<int> cursor = tree.offset;

    for (const Edge& edge : mst) {
        int position = cursor[edge.u]++;
        tree.neighbor[position] = edge.v;
        tree.weight[position] = edge.weight;

        position = cursor[edge.v]++;
        tree.neighbor[position] = edge.u;
        tree.weight[position] = edge.weight;
    }
    return tree;
}

unsigned normalized_thread_count(unsigned requested, std::size_t rows) {
    unsigned threads = requested;
    if (threads == 0) threads = std::thread::hardware_concurrency();
    if (threads == 0) threads = 1;
    if (rows != 0) {
        threads = std::min<unsigned>(threads, static_cast<unsigned>(rows));
    }
    return std::max(1u, threads);
}

template <class Function>
void parallel_row_ranges(std::size_t rows, unsigned requested_threads,
                         Function function) {
    const unsigned threads = normalized_thread_count(requested_threads, rows);
    std::vector<std::thread> workers;
    workers.reserve(threads - 1);

    auto run = [&](unsigned thread_id) {
        const std::size_t begin = rows * thread_id / threads;
        const std::size_t end = rows * (thread_id + 1) / threads;
        function(begin, end);
    };

    for (unsigned thread_id = 0; thread_id + 1 < threads; ++thread_id) {
        workers.emplace_back(run, thread_id);
    }
    // Main participates, so total computation threads equals 'threads'.
    run(threads - 1);
    for (std::thread& worker : workers) worker.join();
}

struct StackItem {
    int vertex;
    int parent;
};

DenseMatrix compute_bottleneck_matrix(const CsrTree& tree,
                                      unsigned requested_threads) {
    if (tree.offset.empty()) return {};
    const int n = static_cast<int>(tree.offset.size()) - 1;
    DenseMatrix bottleneck(static_cast<std::size_t>(n), false);

    parallel_row_ranges(static_cast<std::size_t>(n), requested_threads,
                        [&](std::size_t source_begin,
                            std::size_t source_end) {
        // One allocation per worker, reused for all of its source rows.
        std::vector<StackItem> stack;
        stack.reserve(static_cast<std::size_t>(n));

        for (std::size_t source_index = source_begin;
             source_index < source_end; ++source_index) {
            const int source = static_cast<int>(source_index);
            double* output_row = bottleneck.row(source_index);
            output_row[source] = 0.0;

            stack.clear();
            stack.push_back({source, -1});

            while (!stack.empty()) {
                const StackItem item = stack.back();
                stack.pop_back();
                const int u = item.vertex;
                const double path_maximum = output_row[u];

                for (int position = tree.offset[u];
                     position < tree.offset[u + 1]; ++position) {
                    const int v = tree.neighbor[position];
                    if (v == item.parent) continue;

                    output_row[v] =
                        std::max(path_maximum, tree.weight[position]);
                    stack.push_back({v, u});
                }
            }
        }
    });

    return bottleneck;
}

DenseMatrix create_symmetric_distance_matrix(int n, std::uint32_t seed) {
    if (n <= 0) throw std::invalid_argument("N must be positive");

    DenseMatrix matrix(static_cast<std::size_t>(n), false);
    std::mt19937 generator(seed);
    std::uniform_real_distribution<double> distribution(1.0, 19999.0);

    for (int i = 0; i < n; ++i) {
        matrix(i, i) = 0.0;
        for (int j = i + 1; j < n; ++j) {
            const double value =
                std::round(distribution(generator) * 100.0) / 100.0;
            matrix(i, j) = value;
            matrix(j, i) = value;
        }
    }
    return matrix;
}

}  // namespace reviewer_t5c9
int main(int argc, char* argv[]) {

    int n = 1111;
    for (int i = 1; i < argc; ++i) {
        if (std::string(argv[i]) == "--nodes") {
            if (i + 1 >= argc) {
                std::cerr << "--nodes need to be integer\n";
                return 1;
            }
            try {
                n = std::stoi(argv[++i]);
            } catch (const std::exception&) {
                std::cerr << "--nodes must be integer\n";
                return 1;
            }
        }
    }

    unsigned threads = std::thread::hardware_concurrency();
    constexpr std::uint32_t seed = 2835;

    const unsigned actual_threads =
        reviewer_t5c9::normalized_thread_count(threads, n);

    reviewer_t5c9::DenseMatrix distance =
        reviewer_t5c9::create_symmetric_distance_matrix(n, seed);

    const auto start = reviewer_t5c9::Clock::now();

    std::vector<reviewer_t5c9::Edge> mst =
        reviewer_t5c9::prim_mst_dense(distance);

    distance.release();

    const reviewer_t5c9::CsrTree tree =
        reviewer_t5c9::build_csr_tree(n, mst);

    reviewer_t5c9::DenseMatrix mmj =
        reviewer_t5c9::compute_bottleneck_matrix(tree, actual_threads);

    const auto end = reviewer_t5c9::Clock::now();

    std::cout << std::fixed << std::setprecision(3)
              << "Time used for MMJ matrix (Reviewer t5C9's code cpp version - V2): "
              << reviewer_t5c9::seconds_between(start, end)
              << " s\n";

    std::cout << "Print last 30 values of the first row of mmj matrix:\n"
              << std::setprecision(2);
    const int first = std::max(0, n - 30);
    for (int j = first; j < n; ++j) {
        std::cout << mmj(0, j) << (j + 1 == n ? '\n' : ' ');
    }

    return EXIT_SUCCESS;
}


Writing Reviewer_t5C9_code_cpp_version_V2.cpp


In [ ]:
%%writefile Limited_memory_Variant5_of_Algorithm_13.cpp
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstddef>
#include <cstdint>
#include <cstdlib>
#include <functional>
#include <iomanip>
#include <iostream>
#include <limits>
#include <memory>
#include <numeric>
#include <queue>
#include <random>
#include <stdexcept>
#include <string>
#include <thread>
#include <tuple>
#include <utility>
#include <vector>


// Pipeline:
//   1. Compute an MST of the dense input graph.
//   2. Build a Kruskal reconstruction tree from the MST.
//   3. Put the original vertices in reconstruction-tree leaf order.  Every
//      reconstruction-tree subtree is then a contiguous leaf interval.
//   4. An internal reconstruction-tree node with children A and B says that
//      every pair in A x B has minimax distance equal to that node's weight.
//      Fill those two rectangular matrix regions -- directly at the
//      *original* vertex coordinates, so there is no separate permutation
//      pass afterwards.
//   5. Threads own disjoint leaf-order ranges, which map to disjoint sets of
//      original rows.  Consequently no output lock, atomic operation, or
//      concurrent write to the same cache line is needed in the filling
//      phase.

namespace mmj {

using Clock = std::chrono::steady_clock;

double seconds_between(Clock::time_point a, Clock::time_point b) {
    return std::chrono::duration<double>(b - a).count();
}

class DenseMatrix {
public:
    DenseMatrix() = default;

    // Memory is intentionally uninitialized unless zero_initialize is true.
    // The MMJ filling algorithm writes every off-diagonal entry exactly once
    // and explicitly writes every diagonal entry.
    explicit DenseMatrix(std::size_t n, bool zero_initialize = false)
        : n_(n) {
        if (n_ != 0 && n_ > std::numeric_limits<std::size_t>::max() / n_) {
            throw std::overflow_error("matrix element count overflow");
        }
        const std::size_t count = n_ * n_;
        if (count != 0) {
            data_ = std::unique_ptr<double[]>(new double[count]);
            if (zero_initialize) {
                std::fill_n(data_.get(), count, 0.0);
            }
        }
    }

    DenseMatrix(DenseMatrix&&) noexcept = default;
    DenseMatrix& operator=(DenseMatrix&&) noexcept = default;
    DenseMatrix(const DenseMatrix&) = delete;
    DenseMatrix& operator=(const DenseMatrix&) = delete;

    std::size_t n() const noexcept { return n_; }
    std::size_t elements() const noexcept { return n_ * n_; }

    double* row(std::size_t i) noexcept { return data_.get() + i * n_; }
    const double* row(std::size_t i) const noexcept {
        return data_.get() + i * n_;
    }

    double& operator()(std::size_t i, std::size_t j) noexcept {
        return data_[i * n_ + j];
    }
    const double& operator()(std::size_t i, std::size_t j) const noexcept {
        return data_[i * n_ + j];
    }

    void release() noexcept {
        data_.reset();
        n_ = 0;
    }

private:
    std::size_t n_ = 0;
    std::unique_ptr<double[]> data_;
};

struct Edge {
    int u;
    int v;
    double weight;
};

// Heap-based Prim for a dense matrix.  It scans one dense row for each chosen
// vertex, while the heap avoids a second O(n^2) scan for the next vertex.
std::vector<Edge> prim_mst_dense(const DenseMatrix& distance) {
    const std::size_t n_size = distance.n();
    if (n_size == 0) {
        return {};
    }
    if (n_size > static_cast<std::size_t>(std::numeric_limits<int>::max())) {
        throw std::invalid_argument("too many vertices for 32-bit indices");
    }
    const int n = static_cast<int>(n_size);
    const double inf = std::numeric_limits<double>::infinity();

    std::vector<double> key(n, inf);
    std::vector<int> parent(n, -1);
    std::vector<std::uint8_t> in_mst(n, 0);

    using HeapItem = std::pair<double, int>;
    std::priority_queue<HeapItem, std::vector<HeapItem>,
                        std::greater<HeapItem>> heap;
    key[0] = 0.0;
    heap.emplace(0.0, 0);

    int chosen = 0;
    while (!heap.empty()) {
        const auto [ignored_key, u] = heap.top();
        (void)ignored_key;
        heap.pop();
        if (in_mst[u]) {
            continue;
        }
        in_mst[u] = 1;
        ++chosen;

        const double* drow = distance.row(static_cast<std::size_t>(u));
        for (int v = 0; v < n; ++v) {
            const double w = drow[v];
            if (!in_mst[v] && w < key[v]) {
                key[v] = w;
                parent[v] = u;
                heap.emplace(w, v);
            }
        }
    }

    if (chosen != n) {
        throw std::runtime_error("input graph is disconnected");
    }

    std::vector<Edge> mst;
    mst.reserve(n > 0 ? static_cast<std::size_t>(n - 1) : 0);
    for (int v = 1; v < n; ++v) {
        if (parent[v] < 0) {
            throw std::runtime_error("Prim failed to assign a parent");
        }
        mst.push_back({parent[v], v, distance(v, parent[v])});
    }
    return mst;
}

class DisjointSet {
public:
    explicit DisjointSet(int n)
        : parent_(n), size_(n, 1), tree_root_(n) {
        std::iota(parent_.begin(), parent_.end(), 0);
        std::iota(tree_root_.begin(), tree_root_.end(), 0);
    }

    int find(int x) {
        int root = x;
        while (parent_[root] != root) {
            root = parent_[root];
        }
        while (parent_[x] != x) {
            const int next = parent_[x];
            parent_[x] = root;
            x = next;
        }
        return root;
    }

    int tree_root_of_set(int set_root) const { return tree_root_[set_root]; }

    int unite_roots(int a, int b, int new_tree_root) {
        if (size_[a] < size_[b]) {
            std::swap(a, b);
        }
        parent_[b] = a;
        size_[a] += size_[b];
        tree_root_[a] = new_tree_root;
        return a;
    }

private:
    std::vector<int> parent_;
    std::vector<int> size_;
    std::vector<int> tree_root_;
};

struct ReconstructionNode {
    int left = -1;
    int right = -1;
    double weight = 0.0;
    int begin = -1;  // inclusive leaf-order position
    int end = -1;    // exclusive leaf-order position
};

struct ReconstructionTree {
    int original_vertex_count = 0;
    int root = -1;
    std::vector<ReconstructionNode> nodes;
    // leaf_order[position] = original vertex id
    std::vector<int> leaf_order;
    // position[original vertex id] = position in leaf_order
    std::vector<int> position;
};

ReconstructionTree build_reconstruction_tree(
    int n, std::vector<Edge> mst_edges) {
    if (n <= 0) {
        return {};
    }
    if (static_cast<int>(mst_edges.size()) != n - 1) {
        throw std::invalid_argument("an n-vertex MST must contain n-1 edges");
    }

    std::sort(mst_edges.begin(), mst_edges.end(),
              [](const Edge& a, const Edge& b) {
                  if (a.weight != b.weight) return a.weight < b.weight;
                  if (a.u != b.u) return a.u < b.u;
                  return a.v < b.v;
              });

    ReconstructionTree tree;
    tree.original_vertex_count = n;
    tree.nodes.resize(static_cast<std::size_t>(2 * n - 1));
    tree.leaf_order.resize(n);
    tree.position.resize(n);

    DisjointSet dsu(n);
    int next_node = n;

    for (const Edge& edge : mst_edges) {
        int a = dsu.find(edge.u);
        int b = dsu.find(edge.v);
        if (a == b) {
            throw std::runtime_error("input edge set is not a tree");
        }

        const int left_root = dsu.tree_root_of_set(a);
        const int right_root = dsu.tree_root_of_set(b);
        ReconstructionNode& node = tree.nodes[next_node];
        node.left = left_root;
        node.right = right_root;
        node.weight = edge.weight;

        dsu.unite_roots(a, b, next_node);
        ++next_node;
    }

    tree.root = dsu.tree_root_of_set(dsu.find(0));

    // Iterative postorder traversal: safe even for a completely skewed tree.
    std::vector<std::pair<int, bool>> stack;
    stack.reserve(static_cast<std::size_t>(4 * n));
    stack.emplace_back(tree.root, false);
    int cursor = 0;

    while (!stack.empty()) {
        const auto [node_id, expanded] = stack.back();
        stack.pop_back();
        ReconstructionNode& node = tree.nodes[node_id];

        if (node_id < n) {
            node.begin = cursor;
            node.end = cursor + 1;
            tree.leaf_order[cursor] = node_id;
            tree.position[node_id] = cursor;
            ++cursor;
            continue;
        }

        if (!expanded) {
            stack.emplace_back(node_id, true);
            // Push right first so that left is processed first.
            stack.emplace_back(node.right, false);
            stack.emplace_back(node.left, false);
        } else {
            const ReconstructionNode& left = tree.nodes[node.left];
            const ReconstructionNode& right = tree.nodes[node.right];
            node.begin = left.begin;
            node.end = right.end;
            if (left.end != right.begin) {
                throw std::runtime_error("non-contiguous reconstruction subtree");
            }
        }
    }

    if (cursor != n) {
        throw std::runtime_error("reconstruction tree does not contain all leaves");
    }
    return tree;
}

unsigned normalized_thread_count(unsigned requested, std::size_t n) {
    unsigned threads = requested;
    if (threads == 0) {
        threads = std::thread::hardware_concurrency();
    }
    if (threads == 0) {
        threads = 1;
    }
    if (n != 0) {
        threads = std::min<unsigned>(threads, static_cast<unsigned>(n));
    }
    return std::max(1u, threads);
}

template <class Function>
void parallel_rows(std::size_t n, unsigned requested_threads, Function fn) {
    const unsigned threads = normalized_thread_count(requested_threads, n);
    std::vector<std::thread> workers;
    workers.reserve(threads > 0 ? threads - 1 : 0);

    auto run = [&](unsigned tid) {
        const std::size_t begin = n * tid / threads;
        const std::size_t end = n * (tid + 1) / threads;
        fn(begin, end);
    };

    // The caller thread owns the last partition.  Total active computation
    // threads therefore equals 'threads', not threads+1.
    for (unsigned tid = 0; tid + 1 < threads; ++tid) {
        workers.emplace_back(run, tid);
    }
    run(threads - 1);
    for (std::thread& worker : workers) {
        worker.join();
    }
}

DenseMatrix fill_in_original_order(const ReconstructionTree& tree,
                                    unsigned requested_threads) {
    const int n = tree.original_vertex_count;
    DenseMatrix output(static_cast<std::size_t>(n), false);
    const int* const leaf_order = tree.leaf_order.data();

    parallel_rows(static_cast<std::size_t>(n), requested_threads,
                  [&](std::size_t owned_begin, std::size_t owned_end) {
        // owned_begin/owned_end are a range of *leaf-order positions*, not
        // original vertex ids -- this keeps the work split aligned with the
        // reconstruction tree's contiguous subtree intervals, exactly as in
        // the original code. Each leaf position maps to a distinct original
        // vertex id, so different threads still never write the same row.
        for (int node_id = n; node_id < 2 * n - 1; ++node_id) {
            const ReconstructionNode& node = tree.nodes[node_id];
            const ReconstructionNode& left = tree.nodes[node.left];
            const ReconstructionNode& right = tree.nodes[node.right];

            const std::size_t left_begin = static_cast<std::size_t>(left.begin);
            const std::size_t left_end = static_cast<std::size_t>(left.end);
            const std::size_t right_begin = static_cast<std::size_t>(right.begin);
            const std::size_t right_end = static_cast<std::size_t>(right.end);

            // Owned leaf positions in the left child: fill their row against
            // every original column in the right child's interval.
            const std::size_t lr_begin = std::max(owned_begin, left_begin);
            const std::size_t lr_end = std::min(owned_end, left_end);
            for (std::size_t lp = lr_begin; lp < lr_end; ++lp) {
                double* row_ptr = output.row(
                    static_cast<std::size_t>(leaf_order[lp]));
                for (std::size_t rp = right_begin; rp < right_end; ++rp) {
                    row_ptr[leaf_order[rp]] = node.weight;
                }
            }

            // Owned leaf positions in the right child: fill their row
            // against every original column in the left child's interval.
            const std::size_t rr_begin = std::max(owned_begin, right_begin);
            const std::size_t rr_end = std::min(owned_end, right_end);
            for (std::size_t rp = rr_begin; rp < rr_end; ++rp) {
                double* row_ptr = output.row(
                    static_cast<std::size_t>(leaf_order[rp]));
                for (std::size_t lp = left_begin; lp < left_end; ++lp) {
                    row_ptr[leaf_order[lp]] = node.weight;
                }
            }
        }

        for (std::size_t lp = owned_begin; lp < owned_end; ++lp) {
            const std::size_t orig = static_cast<std::size_t>(leaf_order[lp]);
            output(orig, orig) = 0.0;
        }
    });
    return output;
}

// Kept only for the (rare) case where the caller explicitly wants the matrix
// in reconstruction-tree leaf order rather than original vertex order.
DenseMatrix fill_in_leaf_order(const ReconstructionTree& tree,
                               unsigned requested_threads) {
    const int n = tree.original_vertex_count;
    DenseMatrix output(static_cast<std::size_t>(n), false);

    parallel_rows(static_cast<std::size_t>(n), requested_threads,
                  [&](std::size_t owned_begin, std::size_t owned_end) {
        for (int node_id = n; node_id < 2 * n - 1; ++node_id) {
            const ReconstructionNode& node = tree.nodes[node_id];
            const ReconstructionNode& left = tree.nodes[node.left];
            const ReconstructionNode& right = tree.nodes[node.right];

            const std::size_t left_begin = static_cast<std::size_t>(left.begin);
            const std::size_t left_end = static_cast<std::size_t>(left.end);
            const std::size_t right_begin = static_cast<std::size_t>(right.begin);
            const std::size_t right_end = static_cast<std::size_t>(right.end);

            const std::size_t lr_begin = std::max(owned_begin, left_begin);
            const std::size_t lr_end = std::min(owned_end, left_end);
            for (std::size_t row = lr_begin; row < lr_end; ++row) {
                std::fill(output.row(row) + right_begin,
                          output.row(row) + right_end, node.weight);
            }

            const std::size_t rr_begin = std::max(owned_begin, right_begin);
            const std::size_t rr_end = std::min(owned_end, right_end);
            for (std::size_t row = rr_begin; row < rr_end; ++row) {
                std::fill(output.row(row) + left_begin,
                          output.row(row) + left_end, node.weight);
            }
        }

        for (std::size_t row = owned_begin; row < owned_end; ++row) {
            output(row, row) = 0.0;
        }
    });
    return output;
}

struct Result {
    DenseMatrix matrix;
    ReconstructionTree reconstruction_tree;
    bool in_original_order = false;
};

Result compute_from_mst(int n, const std::vector<Edge>& mst,
                        unsigned requested_threads,
                        bool restore_vertex_order) {
    ReconstructionTree tree = build_reconstruction_tree(n, mst);

    // Single-pass fill: if the caller wants original vertex order (the
    // common case), write directly there and skip the old two-pass
    // fill -> permute pipeline entirely.
    DenseMatrix final_matrix = restore_vertex_order
        ? fill_in_original_order(tree, requested_threads)
        : fill_in_leaf_order(tree, requested_threads);

    return {std::move(final_matrix), std::move(tree), restore_vertex_order};
}

DenseMatrix create_symmetric_distance_matrix(int n, std::uint32_t seed) {
    if (n < 0) {
        throw std::invalid_argument("n must be nonnegative");
    }
    DenseMatrix matrix(static_cast<std::size_t>(n), false);
    std::mt19937 generator(seed);
    std::uniform_real_distribution<double> distribution(1.0, 19999.0);

    for (int i = 0; i < n; ++i) {
        matrix(i, i) = 0.0;
        for (int j = i + 1; j < n; ++j) {
            const double value = std::round(distribution(generator) * 100.0) /
                                 100.0;
            matrix(i, j) = value;
            matrix(j, i) = value;
        }
    }
    return matrix;
}

double value_in_original_numbering(const Result& result, int i, int j) {
    if (result.in_original_order) {
        return result.matrix(i, j);
    }
    const int pi = result.reconstruction_tree.position[i];
    const int pj = result.reconstruction_tree.position[j];
    return result.matrix(pi, pj);
}

}  // namespace mmj

int main(int argc, char* argv[]) {

    int N = 1111;
    for (int i = 1; i < argc; ++i) {
        if (std::string(argv[i]) == "--nodes") {
            if (i + 1 >= argc) {
                std::cerr << "--nodes need to be integer\n";
                return 1;
            }
            try {
                N = std::stoi(argv[++i]);
            } catch (const std::exception&) {
                std::cerr << "--nodes must be integer\n";
                return 1;
            }
        }
    }

    unsigned n_jobs = std::thread::hardware_concurrency();
    if (n_jobs == 0) {
        n_jobs = 1;
    }

    const std::uint32_t random_seed = 2835;
    const bool restore_original_order = true;

    std::cout << "Number of nodes: "
              << N << '\n';

    std::cout << "Number of CPU cores: "
              << mmj::normalized_thread_count(n_jobs, N) << '\n';

    auto distance_matrix =
        mmj::create_symmetric_distance_matrix(
            N,
            random_seed
        );

    const auto start = mmj::Clock::now();

    auto mst =
        mmj::prim_mst_dense(
            distance_matrix
        );

    distance_matrix.release();

    auto result =
        mmj::compute_from_mst(
            N,
            mst,
            n_jobs,
            restore_original_order
        );

    const auto end = mmj::Clock::now();

    const double time_used =
        mmj::seconds_between(
            start,
            end
        );

    std::cout << std::fixed
              << std::setprecision(3);

    std::cout << "Time used for MMJ matrix "
                 "(Limited-memory Variant5 of Algorithm 13): "
              << time_used
              << " seconds\n";

    std::cout << "Print last 30 values of first row "
                 "of MMJ matrix:\n";

    const int first = std::max(0, N - 30);

    for (int j = first; j < N; ++j) {
        std::cout << std::fixed
                  << std::setprecision(2)
                  << mmj::value_in_original_numbering(
                         result,
                         0,
                         j
                     )
                  << " ";
    }

    std::cout << "\n";

    return 0;
}

Writing Limited_memory_Variant5_of_Algorithm_13.cpp


In [ ]:
import numpy as np
import numba
import time
import numpy as np
from sklearn.metrics.pairwise import pairwise_distances



@numba.njit(cache=True, fastmath=True)
def prim_mst(D):
    """
    Optimized Prim's MST using Numba JIT for dense graphs.
    Returns edges as list of (u, v, weight) tuples.
    """
    n = D.shape[0]
    in_mst = np.zeros(n, dtype=np.bool_)
    parent = np.full(n, -1, dtype=np.int64)
    key = np.full(n, np.inf, dtype=D.dtype)
    key[0] = 0.0

    for _ in range(n):
        # Find minimum key vertex not in MST
        u = -1
        min_val = np.inf
        for i in range(n):
            if not in_mst[i] and key[i] < min_val:
                min_val = key[i]
                u = i

        if u == -1:
            break

        in_mst[u] = True

        # Update neighbors
        for v in range(n):
            if not in_mst[v] and D[u, v] < key[v]:
                key[v] = D[u, v]
                parent[v] = u

    # Build edge list
    edges = []
    for v in range(1, n):
        u = parent[v]
        edges.append((u, v, D[u, v]))

    return edges

def build_csr_adjacency(n, mst_edges):
    """Convert MST to compressed sparse row (CSR) format"""

    edge_counts = np.zeros(n, dtype=np.int32)
    for u, v, _ in mst_edges:
        edge_counts[u] += 1
        edge_counts[v] += 1

    ptr = np.zeros(n+1, dtype=np.int32)
    ptr[1:] = np.cumsum(edge_counts)

    adj_edges = np.empty(ptr[-1], dtype=np.int32)
    adj_weights = np.empty(ptr[-1], dtype=np.float64)
    positions = np.zeros(n, dtype=np.int32)

    for u, v, w in mst_edges:
        for _ in range(2):  # Add both directions
            idx = ptr[u] + positions[u]
            adj_edges[idx] = v
            adj_weights[idx] = w
            positions[u] += 1
            u, v = v, u  # Swap for reverse direction

    return ptr, adj_edges, adj_weights

@numba.njit(parallel=True, cache=True, fastmath=True)
def compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights):
    """Numba-optimized BFS for all pairs bottleneck calculation"""
    bottleneck = np.zeros((n, n), dtype=np.float64)

    for src in numba.prange(n):
        visited = np.zeros(n, dtype=numba.boolean)
        max_edges = np.zeros(n, dtype=np.float64)
        queue = np.empty(n, dtype=np.int32)
        queue_weights = np.empty(n, dtype=np.float64)
        front = back = 0

        # Initialize BFS
        visited[src] = True
        queue[back] = src
        queue_weights[back] = 0.0
        back += 1

        while front < back:
            u = queue[front]
            current_max = queue_weights[front]
            front += 1

            # Process all neighbors
            start = ptr[u]
            end = ptr[u+1]
            for i in range(start, end):
                v = adj_edges[i]
                weight = adj_weights[i]

                if not visited[v]:
                    new_max = max(current_max, weight)
                    visited[v] = True
                    max_edges[v] = new_max
                    queue[back] = v
                    queue_weights[back] = new_max
                    back += 1

        bottleneck[src] = max_edges

    return bottleneck

def ultra_fast_wide(distance_matrix):
    n = distance_matrix.shape[0]

    mst = prim_mst(distance_matrix)

    # Convert MST to CSR format
    ptr, adj_edges, adj_weights = build_csr_adjacency(n, mst)

    # Compute bottleneck matrix with Numba
    return compute_bottleneck_matrix(n, ptr, adj_edges, adj_weights)



In [ ]:
import numpy as np
from numba import njit

@njit(cache=True)
def _mt19937_generate_pairs(seed: int, n_pairs: int):
    N_MT = 624
    M    = 397
    MATRIX_A   = np.uint32(0x9908b0df)
    UPPER_MASK = np.uint32(0x80000000)
    LOWER_MASK = np.uint32(0x7fffffff)

    # --- Init state (same formula as original, safe in uint32 under Numba) ---
    mt = np.zeros(N_MT, dtype=np.uint32)
    mt[0] = np.uint32(seed & 0xFFFFFFFF)
    for i in range(1, N_MT):
        prev = mt[i - 1]
        mt[i] = np.uint32(1812433253) * (prev ^ (prev >> np.uint32(30))) + np.uint32(i)

    index  = N_MT                              # triggers generate on first draw
    r0_arr = np.empty(n_pairs, dtype=np.uint32)
    r1_arr = np.empty(n_pairs, dtype=np.uint32)

    # --- Draw 2 × n_pairs uint32s ---
    for k in range(n_pairs * 2):

        # Twist when state is exhausted
        if index >= N_MT:
            for ii in range(N_MT):
                y = (mt[ii] & UPPER_MASK) | (mt[(ii + 1) % N_MT] & LOWER_MASK)
                mt[ii] = mt[(ii + M) % N_MT] ^ (y >> np.uint32(1))
                if y & np.uint32(1):
                    mt[ii] ^= MATRIX_A
            index = 0                          # reset inline — no nonlocal needed

        # Temper
        y = mt[index]
        index += 1
        y ^= (y >> np.uint32(11))
        y ^= (y << np.uint32(7))  & np.uint32(0x9d2c5680)
        y ^= (y << np.uint32(15)) & np.uint32(0xefc60000)
        y ^= (y >> np.uint32(18))

        if k & 1:
            r1_arr[k >> 1] = y   # odd draw → high bits
        else:
            r0_arr[k >> 1] = y   # even draw → low bits

    return r0_arr, r1_arr


def createDistanceMatrix_numba(N: int, seed: int) -> np.ndarray:
    n_pairs = N * (N - 1) // 2
    r0, r1 = _mt19937_generate_pairs(seed, n_pairs)

    # Replicate generate_canonical: r0/2^64 + r1/2^32
    canonical = (r0.astype(np.float64) / 18446744073709551616.0 +
                 r1.astype(np.float64) / 4294967296.0)

    lo, hi = 1.0, 19999.0
    vals = np.round((lo + (hi - lo) * canonical) * 100.0) / 100.0

    A = np.zeros((N, N), dtype=np.float64)
    i_idx, j_idx = np.triu_indices(N, k=1)
    A[i_idx, j_idx] = vals
    A[j_idx, i_idx] = vals
    return A

In [ ]:
%%time

n = 33000
random_seed = 2835

distance_matrix =  createDistanceMatrix_numba(n, random_seed)
distance_matrix =  np.array(distance_matrix)

CPU times: user 38.6 s, sys: 15.1 s, total: 53.7 s
Wall time: 54 s


In [ ]:
# !pip install --upgrade tbb

In [ ]:


print(f"Number of nodes: {n}" )


start = time.time()
mmj_matrix_Reviewer_t5C9_code = ultra_fast_wide(distance_matrix)
end = time.time()
time_used = end - start
time_used = np.round(time_used, 3)

print(f"Time used for MMJ matrix (Reviewer t5C9's code python version): {time_used}s" )
print("Print last 30 values of the first row of MMJ matrix:")
print(mmj_matrix_Reviewer_t5C9_code[0, -30:])


Number of nodes: 33000
Time used for MMJ matrix (Reviewer t5C9's code python version): 25.158s
Print last 30 values of the first row of MMJ matrix:
[1.87 1.88 1.87 1.87 1.87 1.97 1.87 2.52 2.35 1.87 1.87 2.97 1.87 1.87
 1.87 1.87 2.29 1.87 1.87 2.21 2.4  1.87 1.87 1.87 2.31 1.87 1.87 1.87
 1.87 2.26]


In [ ]:
import gc
del distance_matrix
del mmj_matrix_Reviewer_t5C9_code
gc.collect()  # forces immediate collection



44078

In [ ]:
!g++ -std=c++17 -O3  -march=native Reviewer_t5C9_code_cpp_version_V1.cpp  -o tt -ltbb
!./tt --nodes 33000

Number of nodes: 33000
Number of CPU cores: 4
Time used for MMJ matrix (Reviewer t5C9's code cpp version - V1): 36.242 seconds
Print last 30 values of the first row of mmj matrix: 
1.87 1.88 1.87 1.87 1.87 1.97 1.87 2.52 2.35 1.87 1.87 2.97 1.87 1.87 1.87 1.87 2.29 1.87 1.87 2.21 2.40 1.87 1.87 1.87 2.31 1.87 1.87 1.87 1.87 2.26 


In [ ]:
!g++ -std=c++17 -O3 -march=native -flto -DNDEBUG -pthread Reviewer_t5C9_code_cpp_version_V2.cpp -o tt
!./tt --nodes 33000

Time used for MMJ matrix (Reviewer t5C9's code cpp version - V2): 38.888 s
Print last 30 values of the first row of mmj matrix:
1.87 1.88 1.87 1.87 1.87 1.97 1.87 2.52 2.35 1.87 1.87 2.97 1.87 1.87 1.87 1.87 2.29 1.87 1.87 2.21 2.40 1.87 1.87 1.87 2.31 1.87 1.87 1.87 1.87 2.26


In [ ]:
!g++ -std=c++17 -O3 -pthread Variant1_of_Algorithm_13.cpp -o tt
!./tt --nodes 33000

Number of nodes: 33000
Number of CPU cores: 4
Time used for MMJ matrix (new implementation of Variant1 of Algorithm 13):31.348 seconds
Print last 30 values of the first row of mmj matrix:
1.87 1.88 1.87 1.87 1.87 1.97 1.87 2.52 2.35 1.87 1.87 2.97 1.87 1.87 1.87 1.87 2.29 1.87 1.87 2.21 2.40 1.87 1.87 1.87 2.31 1.87 1.87 1.87 1.87 2.26 


In [ ]:
!g++ -std=c++17 -O3 -pthread Variant2_of_Algorithm_13.cpp -o tt
!./tt --nodes 33000

Number of nodes: 33000
Number of CPU cores: 4
Time used for MMJ matrix (new implementation of Variant2 of Algorithm 13):31.917 seconds
Print last 30 values of first row of mmj matrix:
1.87 1.88 1.87 1.87 1.87 1.97 1.87 2.52 2.35 1.87 1.87 2.97 1.87 1.87 1.87 1.87 2.29 1.87 1.87 2.21 2.40 1.87 1.87 1.87 2.31 1.87 1.87 1.87 1.87 2.26 


In [ ]:
!g++ -O3 -std=c++17 -pthread  Limited_memory_Variant5_of_Algorithm_13.cpp -o  tt
!./tt --nodes 33000

Number of nodes: 33000
Number of CPU cores: 4
Time used for MMJ matrix (Limited-memory Variant5 of Algorithm 13): 20.185 seconds
Print last 30 values of first row of MMJ matrix:
1.87 1.88 1.87 1.87 1.87 1.97 1.87 2.52 2.35 1.87 1.87 2.97 1.87 1.87 1.87 1.87 2.29 1.87 1.87 2.21 2.40 1.87 1.87 1.87 2.31 1.87 1.87 1.87 1.87 2.26 


In [ ]:
!g++ -std=c++17 -O3 -march=native -flto -DNDEBUG -pthread Variant5_of_Algorithm_13.cpp -o tt
!./tt --nodes 33000

Number of nodes: 33000
Number of CPU cores: 4
Time used for MMJ matrix (Variant5 of Algorithm 13): 14.949 seconds
Print last 30 values of first row of MMJ matrix:
1.87 1.88 1.87 1.87 1.87 1.97 1.87 2.52 2.35 1.87 1.87 2.97 1.87 1.87 1.87 1.87 2.29 1.87 1.87 2.21 2.40 1.87 1.87 1.87 2.31 1.87 1.87 1.87 1.87 2.26 


In [ ]:
# Time used for calculating the MMJ matrix (33000 nodes):
# 1. Reviewer t5C9's code python version:       25.158 seconds
# 2. Reviewer t5C9's code cpp version V1:       36.242 seconds
# 3. Reviewer t5C9's code cpp version V2:       38.888 seconds
# 4. Variant1 of Algorithm 13:                  31.348 seconds
# 5. Variant2 of Algorithm 13:                  31.917 seconds
# 6. Limited-memory Variant5 of Algorithm 13:   20.185 seconds
# 7. Variant5 of Algorithm 13:                  14.949 seconds

In [ ]:
# According to the experiment results,
# Variant5 of Algorithm 13 is faster than Reviewer t5C9's code in both Python and C++ version.

In [ ]:
import platform
import psutil

# CPU information
print("CPU Information:")
print(f"Processor: {platform.processor()}")
print(f"Physical cores: {psutil.cpu_count(logical=False)}")
print(f"Logical cores: {psutil.cpu_count(logical=True)}")
print(f"CPU frequency: {psutil.cpu_freq().current:.2f} MHz")

# RAM information
ram = psutil.virtual_memory()

print("\nRAM Information:")
print(f"Total RAM: {ram.total / (1024**3):.2f} GB")

CPU Information:
Processor: x86_64
Physical cores: 2
Logical cores: 4
CPU frequency: 2200.00 MHz

RAM Information:
Total RAM: 31.35 GB
